# Assess existing indications from a two-label source diff

This notebook prototypes revision detection without generating latest-label indications. It compares the raw Indications and Usage sections from the label referenced by the current MOAlmanac document and the newest label, then asks one LLM call which existing indications were revised and how.

New and removed indication discovery is intentionally a separate workflow.

In [1]:
import json
import os
from collections import Counter
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

from moalmanac_fda_curation.core.assess_revisions_from_label_diff import (
    assess_revisions_from_label_diff,
    build_revision_assessment_prompt,
    build_section_diff_hunks,
    load_section_pair_from_cache,
)
from moalmanac_fda_curation.core.artifacts import document_label_url
from moalmanac_fda_curation.core.propose_indication_revision import (
    build_label_diff_revision_prompt,
    propose_indication_revision_from_label_diff,
)
from moalmanac_fda_curation.core.reconcile_indications import (
    load_existing_indications,
)

## Configure the Opdivo replay

The baseline is the concrete label URL in the current Opdivo document artifact. The latest URL is selected deterministically from the newest event in the existing Opdivo changelog. Both raw Section 1 snapshots come from the existing cache created by the repository's label extraction code.

In [2]:
env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("No .env file found")
load_dotenv(env_path)
if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(f"ANTHROPIC_API_KEY is missing from {env_path}")

PROJECT_ROOT = Path(env_path).parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
EXISTING_INDICATIONS_JSON = WORKSPACE_ROOT / "moalmanac-db/referenced/indications.json"
BASELINE_DOCUMENT_JSON = PROJECT_ROOT / "analyses/revisions/opdivo-current-record/document.json"
CHANGELOG_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/section1-changelogs/Opdivo-bla125554-section1-changelog.json"
SECTION_CACHE_JSON = WORKSPACE_ROOT / "ai-assisted-gk-curation/analyses/fda-indications/extracted-indications/Opdivo-bla125554-section1-cache.json"

In [3]:
baseline_document = json.loads(BASELINE_DOCUMENT_JSON.read_text())
baseline_label_url = document_label_url(baseline_document)
changelog = json.loads(CHANGELOG_JSON.read_text())
latest_event = max(
    changelog["events"],
    key=lambda event: (event["date"], event["event_number"]),
)
latest_label_url = latest_event["label_url"]

print("Baseline label URL:", baseline_label_url)
print("Latest label date:", latest_event["date"])
print("Latest label URL:", latest_label_url)

Baseline label URL: https://www.accessdata.fda.gov/drugsatfda_docs/label/2025/125554s129lbl.pdf
Latest label date: 2026-03-20
Latest label URL: http://www.accessdata.fda.gov/drugsatfda_docs/label/2026/125554s135lbl.pdf


## Generate the deterministic Section 1 diff

The existing changelog utilities normalize PDF wrapping into logical blocks. The diff includes replacements, insertions, and deletions plus one neighboring block of context on each side. No LLM is involved in this step.

In [4]:
section_pair = load_section_pair_from_cache(
    SECTION_CACHE_JSON,
    baseline_label_url=baseline_label_url,
    latest_label_url=latest_label_url,
)
diff_hunks = build_section_diff_hunks(
    section_pair["baseline_section"],
    section_pair["latest_section"],
    context_blocks=1,
)
print(f"Deterministic diff hunks: {len(diff_hunks)}")

Deterministic diff hunks: 3


In [5]:
for hunk in diff_hunks:
    print("=" * 100)
    print(hunk["hunk_id"], hunk["change_type"].upper())
    print("\nBASELINE CONTEXT BEFORE:", *hunk["baseline_context_before"], sep="\n")
    print("\nBASELINE CHANGED TEXT:", hunk["baseline_text"], sep="\n")
    print("\nLATEST CHANGED TEXT:", hunk["latest_text"], sep="\n")
    print("\nLATEST CONTEXT AFTER:", *hunk["latest_context_after"], sep="\n")

hunk-1 REPLACE

BASELINE CONTEXT BEFORE:
Lung Cancer OPDIVO, in combination with platinum-doublet chemotherapy, is indicated for the neoadjuvant treatment of adult patients with resectable (tumors ≥4 cm or node positive) NSCLC and no known epidermal growth factor receptor (EGFR) mutations or anaplastic lymphoma kinase (ALK) rearrangements, followed by single-agent OPDIVO as adjuvant treatment after surgery.

BASELINE CHANGED TEXT:
• OPDIVO, in combination with ipilimumab, is indicated for the first-line treatment of adult patients with metastatic NSCLC whose tumors express PD-L1 (≥1%) as determined by an FDA-approved test [see Dosage and Administration (2.1)], with no EGFR or ALK genomic tumor aberrations. • OPDIVO, in combination with ipilimumab and 2 cycles of platinum-doublet chemotherapy, is indicated for the first-line treatment of adult patients with metastatic or recurrent NSCLC, with no EGFR or ALK genomic tumor aberrations. • OPDIVO is indicated for the treatment of adult pati

## Load the existing curated indications

In [6]:
existing_indications = load_existing_indications(
    EXISTING_INDICATIONS_JSON, document_id="doc:fda.opdivo"
)
for indication in existing_indications:
    print(f"{indication['id']}: {indication['indication']}\n")

ind:fda.opdivo:0: OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.

ind:fda.opdivo:1: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.

ind:fda.opdivo:2: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic or recurrent non-small cell lung cancer with no EGFR or ALK genomic tumor aberrations as first-line treatment, 

## Inspect the exact assessment prompt

In [7]:
assessment_prompt = build_revision_assessment_prompt(
    existing_indications, diff_hunks
)
print(assessment_prompt)

# Task

Determine which existing curated MOAlmanac FDA indications were revised between
two versions of the label's Indications and Usage section.

The diff hunks below were generated deterministically from the source label text.
Use them as the only evidence of label changes. Associate changes with an existing
indication only when the hunk applies to that indication. A new indication or a
change concerning a different indication is not evidence that the target changed.

# Existing indications

```json
[
  {
    "id": "ind:fda.opdivo:0",
    "indication": "OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery."
  },
  {
    "id": "ind:fda.opdivo:1",
    "indicat

## Assess which existing indications were revised

This is the notebook's only LLM call. It must assess every existing indication and cite the deterministic hunk IDs supporting each revision.

In [8]:
revision_assessment = assess_revisions_from_label_diff(
    existing_indications, diff_hunks
)
print("Verified:", revision_assessment["verified"])
print("Verification errors:", revision_assessment["verification_errors"])
print("Status counts:", Counter(
    item["status"] for item in revision_assessment["assessments"]
))

Verified: True
Verification errors: []
Status counts: Counter({'not_revised': 4, 'revised': 1})


## Review every assessment

In [9]:
for assessment in revision_assessment["assessments"]:
    print("=" * 100)
    print(assessment["existing_indication_id"], assessment["status"].upper())
    print(assessment["existing_indication"]["indication"])
    print("Relevant hunks:", assessment["relevant_hunk_ids"])
    print("Changes:")
    for change in assessment["changes"]:
        print(" -", change)
    print("Reason:", assessment["reason"])

ind:fda.opdivo:0 NOT_REVISED
OPDIVO is a programmed death receptor-1 (PD-1) blocking antibody indicated for the treatment of adult patients with resectable (tumors >=4 cm or node positive) non-small cell lung cancer and no known EGFR mutations or ALK rearrangements, for neoadjuvant treatment, in combination with platinum-doublet chemotherapy, followed by single-agent OPDIVO as adjuvant treatment after surgery.
Relevant hunks: []
Changes:
Reason: No hunks modify the neoadjuvant/adjuvant NSCLC indication. The indication remains unchanged in the context of all hunks.
ind:fda.opdivo:1 REVISED
OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.
Relevant hunks: ['hunk-1']
Changes:
 - Changed 'FDA-approved test' to 'FDA-authoriz

## Inspect the proposal prompt for one revised indication

The assessed `changes` define the proposal's scope. The cited hunk supplies current-label wording, but it may also contain neighboring indications that must not enter the proposal.

In [10]:
revised_assessments = [
    item
    for item in revision_assessment["assessments"]
    if item["status"] == "revised"
]
if not revised_assessments:
    print("No revised indications require proposals.")
else:
    example_assessment = revised_assessments[0]
    proposal_prompt = build_label_diff_revision_prompt(
        example_assessment["existing_indication"],
        example_assessment,
        example_assessment["relevant_hunks"],
    )
    print(proposal_prompt)

# Task

Propose the minimal updates needed for one existing curated MOAlmanac indication
based on its assessed changes between the baseline and current FDA labels.

# Existing editable fields

```json
{
  "indication": "OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.",
  "description": "The U.S. Food and Drug Administration (FDA) granted approval to nivolumab in combination with ipilimumab for the first-line treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations.",
  "raw_biomarkers": "PD-L1 (>= 1%)",
  "raw_cancer_type": "non-small cell lung cancer",
  "raw_therapeutics": "nivolumab in combina

## Propose minimal updates for revised indications

This makes one focused LLM call per revised indication. It may update only `indication`, `description`, and the three `raw_*` fields. Dates, URLs, IDs, and other provenance remain unchanged.

In [11]:
revision_proposals = [
    propose_indication_revision_from_label_diff(
        assessment["existing_indication"],
        assessment,
        diff_hunks,
    )
    for assessment in revised_assessments
]
print(f"Generated proposals: {len(revision_proposals)}")

Generated proposals: 1


In [12]:
for proposal in revision_proposals:
    print("=" * 100)
    print("Existing indication ID:", proposal["existing_indication_id"])
    print("Supporting hunks:", proposal["supporting_hunk_ids"])
    if not proposal["changes"]:
        print("No target-specific field update was proposed.")
    for field, new_value in proposal["changes"].items():
        print(f"\nFIELD: {field}")
        print("EXISTING:", next(
            item["existing_indication"].get(field)
            for item in revised_assessments
            if item["existing_indication_id"] == proposal["existing_indication_id"]
        ))
        print("PROPOSED:", new_value)

Existing indication ID: ind:fda.opdivo:1
Supporting hunks: ['hunk-1']

FIELD: indication
EXISTING: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-approved test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.
PROPOSED: OPDIVO is a programmed death receptor-1 (PD-1)-blocking antibody indicated for the treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1%) as determined by an FDA-authorized test, with no EGFR or ALK genomic tumor aberrations, as first-line treatment in combination with ipilimumab.

FIELD: description
EXISTING: The U.S. Food and Drug Administration (FDA) granted approval to nivolumab in combination with ipilimumab for the first-line treatment of adult patients with metastatic non-small cell lung cancer expressing PD-L1 (>= 1